---------------------------------------------------------

# Script de pré-traitement des images, masques et metadata

### A lancer après l'ajout de nouvelles données.

In [39]:
import pandas as pd
import os
import json
import numpy as np
import pytesseract
from pytesseract import Output
import cv2

## Importation des metadata

In [76]:
# ----------- Gestion des metadata -------------
# Importation du fichier avec les metadatas
df = pd.read_excel('Correspondance_Capture-Echographe.xlsx', sheet_name = 2, usecols=[0,1,4,5,6])
df = df.rename({"id_image": "name_file"}, axis = 1)

df_cap = pd.read_excel('Correspondance_Capture-Echographe.xlsx', sheet_name = 1, usecols=[4,10,11,20])

# Filtre sur les lignes sans identifiants et sans nom
df = df[df.cap_id != "?"]
df = df[df.name_file != "na"]

print(df)
print(df_cap)

      annee  cap_id  long_gonade type_image               name_file
0      2023  422332          7.0        op0  09.34.07 h __[0006800]
1      2023  422332          7.0       op+2  09.34.12 h __[0006801]
2      2023  422332          7.0       op+4  09.34.16 h __[0006802]
3      2023  422332          7.0       op+6  09.34.21 h __[0006803]
4      2023  422332          7.0       op+8  09.34.25 h __[0006804]
...     ...     ...          ...        ...                     ...
1226   2025  443453          5.5       œufs  18.30.25 h __[0008793]
1227   2025  443453          5.5       œufs  18.30.30 h __[0008794]
1228   2025  443453          5.5        op0  18.30.40 h __[0008795]
1229   2025  443453          5.5       op+2  18.31.13 h __[0008797]
1230   2025  443453          5.5       op+4  18.31.19 h __[0008798]

[1220 rows x 5 columns]
       cap_id  cap_lf cap_poids age_referent
0        5653     NaN       NaN          NaN
1        7692    81.0       NaN           0+
2        7693    75.0   

### Conservation uniquement des coupes transversales (avec gonades)

In [77]:
print(f"Nombre d'échographies : {len(df)}")
print(f"Nombre d'échographies de gonades : {len(df[df.type_image != 'œufs'])}")
print(f"Nombre d'échographies d'œufs : {len(df[df.type_image == 'œufs'])}")

Nombre d'échographies : 1220
Nombre d'échographies de gonades : 888
Nombre d'échographies d'œufs : 332


### On récupère l'identifiant des images et on l'ajoute comme une nouvelle colonne

In [78]:
names = []
for name in df.name_file :
    if len(name.split("[")) > 1:
        ref_csv = name.split("[")[1]
        ref_csv = ref_csv.split("]")[0]
        names.append(ref_csv)
    else :
        names.append("NA")

df["image_id"] = names

## Transformation des masques

### Ordonne les classes des masques YOLO : echo transversales

In [79]:
label_dir = "../data/YOLO/labels"

# ----- Ordonne les classes -------
for file_name in os.listdir(label_dir):
    if file_name.endswith(".txt"):
        file_path = os.path.join(label_dir, file_name)
        with open(file_path, "r") as f: # Ouverture du fichier .txt
            lines = f.readlines()

        # Filtre sur les lignes vides
        lines = [line.strip() for line in lines if line.strip()]

        # Ordonne les lignes selon leur classe (premier élément de chaque ligne)
        lines.sort(key=lambda line: int(line.split()[0]))

        # Ajoute un saut de ligne à la fin de chaque ligne
        lines = [line + "\n" for line in lines]

        # Écrire ligne par ligne les nouvelles lignes dans le fichier
        with open(file_path, "w") as f:
            f.writelines(lines)

## Renomage des images et des masques

### Renomage des images et des masques YOLO

In [63]:
def rename(path, format):
    for file_name in os.listdir(path):
        if "-" in file_name : # Vérifie si l'image n'a pas déjà été renomée
            split_name = file_name.split("_")

            if len(split_name) == 4:
                # Récupère du nom l'id de l'image et son heure
                heures = split_name[0]
                cap_id = split_name[2]

                cap_id = cap_id.split("-")[1]
                heures = heures.replace("-",".")
                heures = heures.split(" ")[0]

                try:
                    # Récupère l'année
                    annee = str(df.loc[df.image_id == cap_id, 'annee'].item())
                    # Récupère l'identifiant de la capture (poisson)
                    poisson = str(df.loc[df.image_id == cap_id, 'cap_id'].item())

                    # Nouveau nom
                    new_file_name = f"{cap_id}_{heures}_{poisson}_{annee}.{format}"
                    print(new_file_name)

                except ValueError:
                    # Si cap_id non trouvé ou non unique
                    print(f"L'année n'a pas été trouvée. Le nom n'a pas été changé.")

            # Si déjà renommé avant l'annotation
            if len(split_name) == 5:
                new_file_name = "_".join(split_name[0:4])
                new_file_name = f"{new_file_name.replace("-",".")}.{format}"
                print(new_file_name)

            os.rename(path + file_name, path + new_file_name)

    return True

In [64]:
# Renommages des images
rename("../data/YOLO/images/", "jpg")
rename("../data/YOLO/oeufs/images/", "jpg")
rename("../data/COCO/images/", "jpg")
rename("../data/COCO/oeufs/", "jpg")

# Renommages des labels YOLO
rename("../data/YOLO/labels/", "txt")
rename("../data/YOLO/oeufs/labels/", "txt")

True

### Renomage des masques COCO

In [65]:
def rename_json(json_path, output_json_path):
    if os.path.exists(json_path) :
        # Ouvrir le fichier JSON
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for image in data['images']:
            file_name = image['file_name']
            
            if "-" in file_name:
                split_name = file_name.split("_")

                if len(split_name) == 4:
                    heures = split_name[0]
                    cap_id = split_name[2]
                    
                    cap_id = cap_id.split("-")[1]
                    heures = heures.replace("-", ".")
                    heures = heures.split(" ")[0]
                        
                    try:
                        # Récupère l'année
                        annee = str(df.loc[df.image_id == cap_id, 'annee'].item())
                        # Récupère l'identifiant de la capture (poisson)
                        poisson = str(df.loc[df.image_id == cap_id, 'cap_id'].item())
                        
                        # Nouveau nom
                        new_file_name = f"{cap_id}_{heures}_{poisson}_{annee}.jpg"
                        
                        # Mise à jour de la valeur dans le JSON
                        image['file_name'] = new_file_name
                        
                    except ValueError:
                        # Si cap_id non trouvé ou non unique
                        print(f"L'année n'a pas été trouvée. Le nom n'a pas été changé.")

                # Si déjà renommé avant l'annotation
                if len(split_name) == 5:
                    new_file_name = "_".join(split_name[0:4])
                    new_file_name = f"{new_file_name.replace("-",".")}.jpg"
                    image['file_name'] = new_file_name

        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

    return True

In [34]:
rename_json("../data/COCO/_annotations.coco.json", "../data/COCO/annotations.json")
rename_json("../data/COCO/_annotations.oeufs.coco.json", "../data/COCO/annotations.oeufs.json")

True

### Réduction à 2 classes (suppression de la catégorie "Intestin")

In [80]:
def create_annotations_subset(input_path, output_path, excluded_categories):
    """Crée un fichier COCO en retirant certaines catégories (et leurs annotations)."""
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    excluded_ids = {c['id'] for c in data['categories'] if c['name'] in excluded_categories}

    data['categories'] = [c for c in data['categories'] if c['id'] not in excluded_ids]
    data['annotations'] = [a for a in data['annotations'] if a['category_id'] not in excluded_ids]

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

    return True

input_json_path = "../data/COCO/annotations.json"
output_json_path_2_classes = "../data/COCO/annotations_2_classes.json"

create_annotations_subset(input_json_path, output_json_path_2_classes, excluded_categories=["Intestin"])

True

## Enrichissement du fichier metadata

In [81]:
# ---- Ajout du nouveau nom dans le jeu de données ----
df["new_name_file"] = ""

for index, row in df.iterrows():
    file_name = row["name_file"]
    image_id = row["image_id"]

    split_name = file_name.split("_")
    heures = split_name[0]
    heures = heures.replace("-", ".")
    heures = heures.split(" ")[0]

    annee = str(row['annee'])  # Récupère l'année depuis la ligne actuelle
    poisson = str(row['cap_id'])  # Récupère l'identifiant du poisson depuis la ligne actuelle

    new_name_file = f"{image_id}_{heures}_{poisson}_{annee}"
    df.at[index, "new_name_file"] = new_name_file


In [82]:
# ---- Création de nouvelles colonne pour obtenir la position de l'échographie ----
df['position'] = df[df['type_image'] != "œufs"]['type_image'].str.replace('+', '', regex=False).str[2:].astype(float)
df["position_ratio"] = df.position / df.long_gonade

### Associer une catégorie à chaque échographie

In [ ]:
# Trie les données par poisson et par position pour s'assurer que les échographies sont dans le bon ordre
df = df.sort_values(by=["cap_id", "position"]).reset_index(drop=True)

# Récupère les identifiants uniques des poissons
unique_id = df["cap_id"].unique()

for id in unique_id:
    # Récupère les lignes correspondant au poisson
    all_fish_rows = df["cap_id"] == id
    rows = all_fish_rows & (df['type_image'] != "œufs") # Ignore les écho d'oeufs

    if rows.any():
        first_row = df[rows].index[0] # Récupère la première échographie
        last_row = df[rows].index[-1] # Récupère la dernière échographie

        # 1. Associe une catégorie à chaque image selon la position de l'échographie
        df.loc[rows, "categorie"] = np.where(df.loc[rows, "position_ratio"] > 1, "erreur", 
                                             np.where(df.loc[rows, "position_ratio"] > 0.6, "fin", 
                                                      np.where(df.loc[rows, "position"] > 1, "milieu", "debut"))
                                             )
        
        # 2. Si deux erreurs consécutives détectées, on ne conserve pas le poisson
        if df.loc[rows, "categorie"].eq("erreur").sum() >= 2:
            df = df.drop(df[rows].index)
            continue # Passe au poisson suivant

        # 3. Gestion des petites erreurs
        df.loc[rows, "position"] = np.where(df.loc[rows, "position"] - df.loc[rows, "long_gonade"] < 0, df.loc[rows, "position"],
                                            np.where(df.loc[rows, "position"] - df.loc[rows, "long_gonade"] < 0.5, df.loc[rows, "long_gonade"], df.loc[rows, "position"])
                                            )
        df.loc[rows, "categorie"] = np.where(df.loc[rows, "position"] - df.loc[rows, "long_gonade"] < 0, df.loc[rows, "categorie"],
                                            np.where(df.loc[rows, "position"] - df.loc[rows, "long_gonade"] < 0.5, "fin", df.loc[rows, "categorie"])
                                            )
        
        # 4. Gestion des grosses erreurs (décalage important)
        if (df.loc[rows, "categorie"]=="erreur").any(): # Décale progressif de l'erreur pour chaque échographie (sauf la première) pour essayer d'être le plus proche de la réalité
            progressions = np.arange(0, len(df[rows])) / (len(df[rows])-1)  # Crée un tableau de progression pour chaque échographie
            total_error = df.loc[last_row, "position"] - df.loc[last_row, "long_gonade"] # L'erreur entre la dernière échographie et la longueur de la gonade
            df.loc[rows, "position"] = df.loc[rows, "position"] - (total_error * progressions) # Décale chaque échographie en fonction de sa position
            
            # Mise à jour la position de la dernière échographie pour qu'elle corresponde à la longueur de la gonade
            df.loc[last_row, "position"] = df.loc[last_row, "long_gonade"]

            # Recalcule le ratio après le décalage
            df.loc[rows, "position_ratio"] = df.loc[rows, "position"] / df.loc[rows, "long_gonade"]

            # Mise à jour des catégories après le décalage
            df.loc[rows, "categorie"] = np.where(df.loc[rows, "position_ratio"] > 1, "erreur", 
                                                 np.where(df.loc[rows, "position_ratio"] > 0.6, "fin", 
                                                          np.where(df.loc[rows, "position"] > 1, "milieu", "debut"))
                                                          )

        # 5. Supprime les doublons de positions pour un même poisson (sauf oeufs)
        duplicates = df.loc[rows].duplicated(subset=["cap_id", "position"], keep="last")
        df = df.drop(df[rows].index[duplicates])

        # 6. Regroupe les catégories dans une nouvelle colonne
        categories = str(list(df.loc[rows, "categorie"]))
        df.loc[rows, "categories"] = categories

# 7. Supprime les lignes vides
df = df.dropna(how = 'all')
df

,annee,cap_id,long_gonade,type_image,name_file,image_id,new_name_file,position,position_ratio,categorie,categories
0,2019,356282,5.0,op+1,10.13.22 h __[0004569],0004569,0004569_10.13.22_356282_2019,1.0,0.200000,debut,"['debut', 'milieu']"
1,2019,356282,5.0,op+3,10.13.22 h __[0004570],0004570,0004570_10.13.22_356282_2019,3.0,0.600000,milieu,"['debut', 'milieu']"
2,2019,356282,5.0,œufs,10.14.11 h __[0004571],0004571,0004571_10.14.11_356282_2019,NaN,NaN,NaN,NaN
3,2019,356302,7.1,op+1,10.29.35 h __[0004572],0004572,0004572_10.29.35_356302_2019,1.0,0.140845,debut,"['debut', 'milieu', 'milieu']"
4,2019,356302,7.1,op+3,10.30.05 h __[0004573],0004573,0004573_10.30.05_356302_2019,3.0,0.422535,milieu,"['debut', 'milieu', 'milieu']"
...,...,...,...,...,...,...,...,...,...,...,...
1215,2025,444313,4.5,op0,14.26.39 h __[0009134],0009134,0009134_14.26.39_444313_2025,0.0,0.000000,debut,"['debut', 'milieu', 'fin']"
1216,2025,444313,4.5,op+2,14.27.26 h __[0009136],0009136,0009136_14.27.26_444313_2025,2.0,0.444444,milieu,"['debut', 'milieu', 'fin']"
1217,2025,444313,4.5,op+4,14.27.31 h __[0009137],0009137,0009137_14.27.31_444313_2025,4.0,0.888889,fin,"['debut', 'milieu', 'fin']"
1218,2025,444313,4.5,œufs,14.26.15 h __[0009132],0009132,0009132_14.26.15_444313_2025,NaN,NaN,NaN,NaN


In [140]:
print(f"Nombre d'images incohérentes : {len(df[df['categorie'] == 'erreur'])}")
print(f"Nombre d'images au début : {len(df[df['categorie'] == 'debut'])}") 
print(f"Nombre d'images au milieu : {len(df[df['categorie'] == 'milieu'])}") 
print(f"Nombre d'images à la fin : {len(df[df['categorie'] == 'fin'])}") 
print(f"Nombre de combinaisons différentes : {len(df['categories'].value_counts())}")
print(f"Nombre de poissons différents : {len(df['cap_id'].value_counts())}")

df_unique = df[~df.duplicated(subset=["cap_id"], keep = 'first')] # Garde qu'une ligne par poisson
print(df_unique["categories"].value_counts())

Nombre d'images incohérentes : 0
Nombre d'images au début : 242
Nombre d'images au milieu : 361
Nombre d'images à la fin : 269
Nombre de combinaisons différentes : 20
Nombre de poissons différents : 243
categories
['debut', 'milieu', 'fin']                                         73
['debut', 'milieu', 'milieu', 'fin']                               54
['debut', 'milieu', 'milieu', 'fin', 'fin']                        25
['debut', 'milieu']                                                20
['debut', 'milieu', 'fin', 'fin']                                  14
['debut', 'milieu', 'milieu', 'milieu', 'fin']                     12
['debut', 'fin']                                                   11
['debut', 'milieu', 'milieu', 'milieu', 'fin', 'fin']              10
['debut', 'milieu', 'milieu']                                       6
['debut', 'fin', 'fin']                                             3
['debut', 'debut', 'fin']                                           3
['milieu', 'mili

### Ajouter les informations du poisson

In [141]:
id_unique = df_unique["cap_id"]

for id in id_unique:
    # Récupère la ligne correspondant au poisson
    row = df_cap[df_cap.cap_id == id]

    # Complète df avec le poids, la taile et l'age du poisson
    df.loc[df.cap_id == id, "poids_poisson"] = float(str(row["cap_poids"].values[0]).replace(",", "."))
    df.loc[df.cap_id == id, "long_poisson"] = float(row["cap_lf"].values[0])
    df.loc[df.cap_id == id, "age_poisson"] = row["age_referent"].values[0]


/tmp/user/1001/ipykernel_873368/2062193815.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2+' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df.cap_id == id, "age_poisson"] = row["age_referent"].values[0]


### Récupère l'échelle de l'échographie

In [143]:
# ---- Récupération de l'échelle par OCR ----
def OCR(image_path):
    image = cv2.imread(image_path)

    # Crop et redimensionnement de l'image
    x, y, w, h = 530, 350, 250, 200  # crop left, crop top, size right, size bottom
    new_width, new_height = 280, 300
    cropped_img = image[y:y+h, x:x+w] # crop
    resized_img = cv2.resize(cropped_img, (new_width, new_height)) # Redimensionne
    # Save resized image for debugging
    cv2.imwrite("resized_img.jpg", resized_img)

    text = pytesseract.image_to_string(resized_img,lang='eng').strip()

    return text

In [146]:
for id in id_unique :
    echos = df[df.cap_id.isin([id])]
    echelle = 0

    for index,row in echos.iterrows():
        file_path = row["new_name_file"]
        image_path = f"../data/YOLO/oeufs/images/{file_path}.jpg" # "../data/YOLO/images/{file_path}.jpg" ou "../data/YOLO/oeufs/images/{file_path}.jpg"

        if os.path.exists(image_path):
            # --- Récupération de l'échelle par OCR ---
            ocr = OCR(image_path)
            if ocr == "3,8":
                echelle = float(ocr.replace(",",".").split()[0])
                continue
            elif ocr == "4.7":
                echelle = float(ocr.replace(",",".").split()[0])
                continue
            elif echelle != 3.8 and echelle != 4.7:
                echelle = 3.1

    if echelle == 3.8:
        df.loc[df.cap_id == id, "echelle"] = 3.8
    if echelle == 4.7:
        df.loc[df.cap_id == id, "echelle"] = 4.7
    if echelle == 3.1:
        df.loc[df.cap_id == id, "echelle"] = 3.1


### Sauvegarde des jeux de données finaux

In [148]:
# Fichier final contenant toutes les informations sur les images
df_final = df
df_final.to_excel("correspondance_echo_final.xlsx", index=False)

df_final

,annee,cap_id,long_gonade,type_image,name_file,image_id,new_name_file,position,position_ratio,categorie,categories,poids_poisson,long_poisson,age_poisson,echelle
0,2019,356282,5.0,op+1,10.13.22 h __[0004569],0004569,0004569_10.13.22_356282_2019,1.0,0.200000,debut,"['debut', 'milieu']",74.9,186.0,NaN,NaN
1,2019,356282,5.0,op+3,10.13.22 h __[0004570],0004570,0004570_10.13.22_356282_2019,3.0,0.600000,milieu,"['debut', 'milieu']",74.9,186.0,NaN,NaN
2,2019,356282,5.0,œufs,10.14.11 h __[0004571],0004571,0004571_10.14.11_356282_2019,NaN,NaN,NaN,NaN,74.9,186.0,NaN,NaN
3,2019,356302,7.1,op+1,10.29.35 h __[0004572],0004572,0004572_10.29.35_356302_2019,1.0,0.140845,debut,"['debut', 'milieu', 'milieu']",65.6,176.0,NaN,3.1
4,2019,356302,7.1,op+3,10.30.05 h __[0004573],0004573,0004573_10.30.05_356302_2019,3.0,0.422535,milieu,"['debut', 'milieu', 'milieu']",65.6,176.0,NaN,3.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215,2025,444313,4.5,op0,14.26.39 h __[0009134],0009134,0009134_14.26.39_444313_2025,0.0,0.000000,debut,"['debut', 'milieu', 'fin']",73.1,187.0,.,NaN
1216,2025,444313,4.5,op+2,14.27.26 h __[0009136],0009136,0009136_14.27.26_444313_2025,2.0,0.444444,milieu,"['debut', 'milieu', 'fin']",73.1,187.0,.,NaN
1217,2025,444313,4.5,op+4,14.27.31 h __[0009137],0009137,0009137_14.27.31_444313_2025,4.0,0.888889,fin,"['debut', 'milieu', 'fin']",73.1,187.0,.,NaN
1218,2025,444313,4.5,œufs,14.26.15 h __[0009132],0009132,0009132_14.26.15_444313_2025,NaN,NaN,NaN,NaN,73.1,187.0,.,NaN
